# The refusal boundary

**What this notebook is.** A working tour of the line between what this data
supports and what it does not, run live rather than described. Task 09
generated 436 candidate claims from the committed verdict tables and put each
one through four gates; 316 were refused. This notebook opens the gates,
sends sentences through them, and shows what happens.

It is the notebook to have open in a mentor review. "Why can't you say X?" has
an executable answer here: type X into the gates and watch which one stops it,
and where that gate's authority came from.

**What it reads.** [`src/insights.py`](../src/insights.py) and the committed
Task 09 tables. No data files, no keys, no network.

In [1]:
import sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parent
sys.path.insert(0, str(REPO / "src"))

import insights as ins

MEMBER = REPO / "members" / "ankit-google"
TASK09 = MEMBER / "task-09-tables"

ledger = pd.read_csv(TASK09 / "claim-ledger.csv")
print(f"{len(ledger)} claims in the ledger")
print(ledger.status.value_counts().to_string())

436 claims in the ledger
status
refused                316
published              102
published_qualified     18


## 1. Four gates, in order

A claim is a record, not a sentence, and it must clear four gates before it may
be spoken:

| Gate | Question it asks |
| --- | --- |
| evidence | Does a committed table contain this number? |
| lint | Does the wording assert something the metric cannot carry? |
| identification | Did an upstream task settle this, or leave it open? |
| consistency | Does this contradict a claim already published? |

The gate that stops a claim is recorded, and the distribution is the whole
argument of Task 09: refusals are overwhelmingly about **what the data can
carry**, not about how a sentence is phrased.

In [2]:
refused = ledger[ledger.status == ins.REFUSED]
print(refused.blocked_by.value_counts().to_string())
print()
share = (refused.blocked_by == "identification").mean()
print(f"{share:.1%} of refusals are identification, not wording")

blocked_by
identification    306
lint                9
evidence            1

96.8% of refusals are identification, not wording


## 2. The language gate, run live

`ins.lint_text` is the wording gate. Nine rules: eight fixed patterns from
`ins.PROHIBITED_PATTERNS`, plus a ninth built from the country vocabulary the
data actually contains — passed in as an argument, because a country name is
data, not a constant.

Below are six sentences a strategy team would like to have. Each is a
paraphrase of something the ledger has an opinion about.

In [3]:
countries = ins.country_vocabulary(MEMBER)
print(f"{len(countries)} country terms in the vocabulary: {countries[:6]}…")

tempting = [
    "Google's hiring grew 12% in 2023.",
    "Google is hiring more engineers than Meta.",
    "We forecast Google's postings to rise next quarter.",
    "Google and Meta are converging on the same skill mix.",
    "Hiring in Germany looks different from the UK.",
    "Google pays more for ML engineers than Snowflake.",
]

rows = []
for line in tempting:
    hits = ins.lint_text(line, countries)
    rows.append({
        "sentence": line,
        "rules": ", ".join(hit["rule"] for hit in hits) or "—",
        "wording gate": "refused" if hits else "clean",
    })
pd.DataFrame(rows)

48 country terms in the vocabulary: ('Argentina', 'Australia', 'Austria', 'Belgium', 'Brazil', 'Canada')…


,sentence,rules,wording gate
0,Google's hiring grew 12% in 2023.,—,clean
1,Google is hiring more engineers than Meta.,—,clean
2,We forecast Google's postings to rise next qua...,forecast,refused
3,Google and Meta are converging on the same ski...,convergence,refused
4,Hiring in Germany looks different from the UK.,country_split,refused
5,Google pays more for ML engineers than Snowflake.,—,clean


**Three of the six pass.** That is the result worth having in front of a
mentor, and it is not a defect being confessed — it is the design. Read the
three that were caught first:

In [4]:
for line in tempting:
    for hit in ins.lint_text(line, countries):
        print(f"{hit['rule']:<22} matched {hit['matched']!r}")
        print(f"{'':<22} source: {hit['source']}")
        print(f"{'':<22} why: {hit['why']}")
        print()

forecast               matched 'next quarter'
                       source: task-07 §8
                       why: no model beats persistence and the h=1 interval spans 3.15x, so the maximum useful horizon is 0

convergence            matched 'converging'
                       source: task-08 §8
                       why: trajectory similarity is refused: 1 of 15 pairs is eligible and the mean correlation sits inside the closure null

country_split          matched 'Germany'
                       source: task-05 §9
                       why: publishers are regional, so a country figure compares aggregator footprints, not hiring



### 2.1 The three it misses

Each rule is a regular expression over a finite vocabulary, so each one has an
edge. The clearest case is the level comparison. The ledger's own phrasing of
that claim is caught; the paraphrase above is not, because *engineers* is not
in the rule's noun list and *jobs* is:

In [5]:
level = ledger[ledger.claim_id == "tempting-level"].iloc[0]
mine = "Google is hiring more engineers than Meta."

for label, text in (("ledger's phrasing", level.text), ("paraphrase", mine)):
    hits = [hit["rule"] for hit in ins.lint_text(text, countries)]
    print(f"{label:<18} {text[:78]}")
    print(f"{'':<18} caught by: {hits or 'nothing'}")

ledger's phrasing  Google posts more jobs than Snowflake, at 0.2403 of the common panel
                   caught by: ['cross_company_level']
paraphrase         Google is hiring more engineers than Meta.
                   caught by: nothing


The comment above that rule in `src/insights.py` says as much — *"a rule that
catches 'more jobs than' and not 'more roles than' is a rule a writer clears by
accident."* The noun list is as long as the phrasings this project actually
produced, and no longer. A writer reaching for a synonym clears it.

Which is exactly why the wording gate is not the boundary. It is a fast floor
that catches the phrasings this project is known to reach for. The boundary is
identification, and every sentence the wording gate misses is refused there
instead. Below, each clean sentence is matched to the ledger row that actually
stops it:

In [6]:
covers = {
    "Google's hiring grew 12% in 2023.": "vol-google",
    "Google is hiring more engineers than Meta.": "vol-google",
    "Google pays more for ML engineers than Snowflake.":
        "salary-google-snowflake-engineering",
}
clean = [line for line in tempting if not ins.lint_text(line, countries)]
assert set(clean) == set(covers), "the wording gate's misses have moved"

for line in clean:
    row = ledger[ledger.claim_id == covers[line]].iloc[0]
    print(line)
    print(f"    stopped by : {row.claim_id} — {row.status}, at the "
          f"{row.blocked_by} gate")
    print(f"    the claim  : {row.text[:96]}")
    print(f"    falsifier  : {row.falsifier}")
    print()

Google's hiring grew 12% in 2023.
    stopped by : vol-google — refused, at the identification gate
    the claim  : Google's 2023 posting volume reads as growth, and the four panel treatments spread 99.05 index p
    falsifier  : a twelve-month panel in which all four treatments agree on the sign

Google is hiring more engineers than Meta.
    stopped by : vol-google — refused, at the identification gate
    the claim  : Google's 2023 posting volume reads as growth, and the four panel treatments spread 99.05 index p
    falsifier  : a twelve-month panel in which all four treatments agree on the sign

Google pays more for ML engineers than Snowflake.
    stopped by : salary-google-snowflake-engineering — refused, at the identification gate
    the claim  : In Engineering roles on via Ai-Jobs.net, Google's median disclosed salary differs from Snowflake
    falsifier  : the same sign on a publisher-balanced sample of disclosed salaries



The growth sentence and the comparison both need Google's 2023 level, and
`vol-google` is refused: four defensible panel treatments spread 99.05 index
points around it, so the sign is not identified. The pay sentence needs a
salary gap, and every one of the 15 `salary_gap` rows is refused — Google
discloses a salary in 4.02% of its 2023 postings, and a median over 4% of
postings is a median over whoever chose to disclose.

Note what this costs a paraphrase: nothing. The wording gate can be walked
around by a synonym, and the claim is still refused, because the refusal lives
in the ledger row rather than in the sentence.

### 2.2 The negation problem

A qualifying clause earns its place by **naming the forbidden reading in order
to deny it** — "not headcount", "not absolute volume". So a sentence that
passed the gate in Task 09 fails it when re-linted, on the very words that made
it publishable. Here is a real published claim, rendered from the ledger:

In [7]:
share_claim = ledger[ledger.claim_id == "share-google"].iloc[0]
rendered = ins.sentence(share_claim)
print(rendered)
print()
print("ledger records gate_lint:", share_claim.gate_lint)
print("re-linting it now fires:",
      [hit["rule"] for hit in ins.lint_text(rendered, countries)])

Google was losing share of the shared publisher pool between H1 and H2 2023 (log share change -0.182), agreeing in 4 of 6 publishers — share of a fixed publisher pool, not headcount and not absolute volume; unanimity here is floor-dependent, see C8

ledger records gate_lint: True
re-linting it now fires: ['unmeasured_construct']


`unmeasured_construct`, matched on the word *headcount* — which appears only
because the clause is denying it.

This is not a bug to suppress with an exception list. It is a fact about
negation, and [`src/present.py`](../src/present.py) is built around it: a deck
bullet bound to a ledger row is **exempt** from the language gate, because that
gate already ran on the sentence once, in Task 09, and the ledger records the
verdict in `gate_lint`. The linter checks that recorded verdict instead. Only
the words Task 10 itself wrote get linted.

## 3. The identification gate: where the refusals actually come from

Nothing in the wording of "Google's posting volume rose in 2023" is wrong. It
is refused because Task 05 could not settle the direction — four defensible
panel treatments disagree — and no phrasing recovers a verdict the upstream
analysis does not have.

In [8]:
refusals = pd.read_csv(TASK09 / "refused-claims.csv")
identification = refusals[refusals.blocked_by == "identification"]
print(f"{len(identification)} claims refused at identification\n")
for _, row in identification.head(4).iterrows():
    print(f"[{row.claim_id}] {row.text[:110]}…")
    print(f"    what would lift it: {row.what_would_lift_it}")
    print()

306 claims refused at identification

[vol-databricks] Databricks's 2023 posting volume reads as unresolved, and the four panel treatments spread 128.10 index points…
    what would lift it: more months, more publishers, or a stratum with enough support to settle the upstream verdict

[vol-google] Google's 2023 posting volume reads as growth, and the four panel treatments spread 99.05 index points around i…
    what would lift it: more months, more publishers, or a stratum with enough support to settle the upstream verdict

[vol-microsoft] Microsoft's 2023 posting volume reads as growth, and the four panel treatments spread 80.93 index points aroun…
    what would lift it: more months, more publishers, or a stratum with enough support to settle the upstream verdict

[vol-nvidia] Nvidia's 2023 posting volume reads as decline, and the four panel treatments spread 27.38 index points around …
    what would lift it: more months, more publishers, or a stratum with enough support to settle t

## 4. What a published claim looks like

A published claim carries its clause. `ins.sentence` renders the two together,
and the clause is not decoration: it is the part that makes the sentence true.
The `published_qualified` claims are false without it.

In [9]:
qualified = ledger[ledger.status == ins.QUALIFIED]
print(f"{len(qualified)} qualified claims; {len(ledger[ledger.status == ins.PUBLISHED])} "
      f"published outright\n")
for _, row in qualified.head(3).iterrows():
    print(f"[{row.claim_id}]")
    print(f"  {ins.sentence(row)}")
    print()

18 qualified claims; 102 published outright

[vol-meta]
  Meta's 2023 posting volume reads as growth, and the four panel treatments spread 135.59 index points around it — measured on the shared publisher panel only

[vol-snowflake]
  Snowflake's 2023 posting volume reads as decline, and the four panel treatments spread 17.24 index points around it — measured on the shared publisher panel only

[share-databricks]
  Databricks was gaining share of the shared publisher pool between H1 and H2 2023 (log share change 0.293), agreeing in 4 of 6 publishers — share of a fixed publisher pool, not headcount and not absolute volume; unanimity here is floor-dependent, see C8



In [10]:
# The clause on every relative-share claim names the correction it depends on.
shares = ledger[(ledger.family == "relative_share") &
                (ledger.status.str.startswith("published"))]
print(shares[["claim_id", "status"]].to_string(index=False))
print()
print("every one carries C8:", shares.clause.str.contains("C8").all())

        claim_id              status
share-databricks published_qualified
    share-google published_qualified
      share-meta published_qualified
 share-microsoft published_qualified
    share-nvidia published_qualified
 share-snowflake published_qualified

every one carries C8: True


## 5. What would change the answer

Every claim in the ledger names a falsifier, on both sides of the line. For a
published claim it is the observation that would overturn it; for a refusal it
is the observation that would lift it. This is the difference between a
limitation and an excuse: a limitation says what would have to be true instead.

[`falsifiers.csv`](../members/ankit-google/task-09-tables/falsifiers.csv) holds
the 120 publishable claims. The refusals carry theirs in the ledger, and in
`what_would_lift_it` in [`refused-claims.csv`](../members/ankit-google/task-09-tables/refused-claims.csv).

In [11]:
falsifiers = pd.read_csv(TASK09 / "falsifiers.csv")
print(f"{len(falsifiers)} publishable claims, each with a falsifier\n")
for _, row in falsifiers.head(3).iterrows():
    print(f"[{row.claim_id}] {row.sentence[:86]}…")
    print(f"    overturned by: {row.falsifier}")
    print()

print("--- and the refusals name what would lift them ---\n")
for _, row in refused.head(3).iterrows():
    print(f"[{row.claim_id}] lifted by: {row.falsifier}")

120 publishable claims, each with a falsifier

[vol-meta] Meta's 2023 posting volume reads as growth, and the four panel treatments spread 135.5…
    overturned by: a twelve-month panel in which all four treatments agree on the sign

[vol-snowflake] Snowflake's 2023 posting volume reads as decline, and the four panel treatments spread…
    overturned by: a twelve-month panel in which all four treatments agree on the sign

[share-databricks] Databricks was gaining share of the shared publisher pool between H1 and H2 2023 (log …
    overturned by: the same sign count under a per-company cell floor of 5 postings a half

--- and the refusals name what would lift them ---

[vol-databricks] lifted by: a twelve-month panel in which all four treatments agree on the sign
[vol-google] lifted by: a twelve-month panel in which all four treatments agree on the sign
[vol-microsoft] lifted by: a twelve-month panel in which all four treatments agree on the sign


## 6. Self-check

The same rule as everywhere else in this repository: a notebook that quotes a
number asserts it, so a rebuild that moves the number fails here instead of
leaving a confident sentence standing.

In [12]:
caught = {hit["rule"] for line in tempting for hit in ins.lint_text(line, countries)}

checks = {
    "436 claims in the ledger": len(ledger) == 436,
    "316 refused": len(refused) == 316,
    "most refusals are identification, not wording": share > 0.9,
    "nine language rules run": len(ins.PROHIBITED_PATTERNS) + 1 == 9,
    "the wording gate catches three of the six": len(clean) == 3,
    "and catches them on forecast, convergence and country_split":
        caught == {"forecast", "convergence", "country_split"},
    "the ledger's own phrasing of the level claim is caught": any(
        hit["rule"] == "cross_company_level"
        for hit in ins.lint_text(level.text, countries)),
    "every sentence the wording gate misses is refused elsewhere": all(
        ledger[ledger.claim_id == cid].iloc[0].status == ins.REFUSED
        for cid in covers.values()),
    "no salary gap is published": not len(
        ledger[(ledger.family == "salary_gap") &
               (ledger.status != ins.REFUSED)]),
    "a published claim trips the gate its own clause satisfies": bool(
        ins.lint_text(rendered, countries)) and bool(share_claim.gate_lint),
    "every relative-share claim carries C8": bool(shares.clause.str.contains("C8").all()),
    "every claim names a falsifier, published or refused":
        bool(ledger.falsifier.notna().all()) and len(falsifiers) == 120,
    "no personal-data column in the ledger": not ins.personal_data_columns_present(ledger),
}
for label, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'} — {label}")
assert all(checks.values()), "this notebook is stale against src/insights.py"

PASS — 436 claims in the ledger
PASS — 316 refused
PASS — most refusals are identification, not wording
PASS — nine language rules run
PASS — the wording gate catches three of the six
PASS — and catches them on forecast, convergence and country_split
PASS — the ledger's own phrasing of the level claim is caught
PASS — every sentence the wording gate misses is refused elsewhere
PASS — no salary gap is published
PASS — a published claim trips the gate its own clause satisfies
PASS — every relative-share claim carries C8
PASS — every claim names a falsifier, published or refused
PASS — no personal-data column in the ledger
